# 06 - Análise de Internações por Doenças Respiratórias

Este notebook investiga a demanda hospitalar relacionada a doenças respiratórias em Goiás, utilizando a base consolidada do SIH/SUS e a população municipal, para responder perguntas de negócio do grupo de problemas respiratórios.

## 1. Configuração inicial

In [ ]:
import pandas as pd


pd.set_option("display.max_columns", None)

## 2. Carregamento das bases consolidadas

In [4]:
df_sih = pd.read_parquet("../data/silver/sih_multianual.parquet")
df_populacao = pd.read_parquet("../data/silver/populacao_multianual.parquet")

print("SIH:", df_sih.shape)
print("População:", df_populacao.shape)

SIH: (2286755, 19)
População: (33422, 10)


## 3. Preparação e regras de negócio

Esta análise reutiliza as mesmas regras de negócio definidas no notebook `05_analise_integrada.ipynb`, para manter consistência com as demais análises do projeto:

- São consideradas apenas as AIHs regulares (`IDENT = 1`). AIHs de longa permanência (`IDENT = 5`) não são contabilizadas como novas internações.
- O recorte temporal considerado é de janeiro de 2021 a junho de 2026, com base na data de internação (`DT_INTER`).
- Para os indicadores por população, são consideradas apenas as internações de residentes em municípios de Goiás.

In [5]:
df_sih["AIH_LONGA_PERMANENCIA"] = df_sih["IDENT"] == "5"
df_sih["DT_INTER_DATA"] = pd.to_datetime(df_sih["DT_INTER"])

df_sih_regulares = df_sih[~df_sih["AIH_LONGA_PERMANENCIA"]].copy()

inicio_analise = pd.Timestamp("2021-01-01")
fim_analise = pd.Timestamp("2026-06-30")

df_sih_demanda = df_sih_regulares[
    df_sih_regulares["DT_INTER_DATA"].between(inicio_analise, fim_analise)
].copy()

df_sih_demanda["ANO_INTER"] = df_sih_demanda["DT_INTER_DATA"].dt.year
df_sih_demanda["MES_INTER"] = df_sih_demanda["DT_INTER_DATA"].dt.month
df_sih_demanda["MUNIC_RES"] = df_sih_demanda["MUNIC_RES"].astype("string").str.zfill(6)

print("AIHs regulares no período de análise:", len(df_sih_demanda))

AIHs regulares no período de análise: 2217182


## 4. Identificação das internações respiratórias

As internações respiratórias são identificadas a partir do diagnóstico principal (`DIAG_PRINC`), considerando os códigos da CID-10 que iniciam com `J` (Capítulo X - Doenças do aparelho respiratório, J00 a J99).

Essa é uma regra de negócio nova, específica desta análise, e ainda não utilizada em nenhum outro notebook do projeto.

In [6]:
df_sih_demanda["DIAG_PRINC"] = df_sih_demanda["DIAG_PRINC"].astype("string")

df_sih_demanda["RESPIRATORIO"] = (
    df_sih_demanda["DIAG_PRINC"]
    .str.upper()
    .str.startswith("J", na=False)
)

df_sih_demanda["RESPIRATORIO"].value_counts()

RESPIRATORIO
False    2020203
True      196979
Name: count, dtype: Int64

### 4.1. Volume e participação no total de internações

In [8]:
total_internacoes = len(df_sih_demanda)
total_respiratorias = df_sih_demanda["RESPIRATORIO"].sum()

print("Internações totais no período:", total_internacoes)
print("Internações respiratórias no período:", total_respiratorias)
print(
    "Participação das internações respiratórias no total: {:.2f}%".format(
        total_respiratorias / total_internacoes * 100
    )
)

Internações totais no período: 2217182
Internações respiratórias no período: 196979
Participação das internações respiratórias no total: 8.88%


### 4.2. Diagnósticos respiratórios mais frequentes

In [9]:
df_sih_demanda[df_sih_demanda["RESPIRATORIO"]]["DIAG_PRINC"].value_counts().head(10)

DIAG_PRINC
J189    49396
J180    14200
J159    13874
J18      8274
J459     6544
J449     6251
J158     6039
J353     5288
J441     4441
J960     4167
Name: count, dtype: Int64

## 5. Sazonalidade das internações respiratórias

A distribuição mensal é analisada considerando apenas os anos completos (2021 a 2025), para não distorcer a sazonalidade com o período parcial de 2026.

In [10]:
df_anos_completos = df_sih_demanda[df_sih_demanda["ANO_INTER"].between(2021, 2025)]

sazonalidade = (
    df_anos_completos
    .groupby(["MES_INTER", "RESPIRATORIO"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "internacoes_nao_respiratorias", True: "internacoes_respiratorias"})
)

sazonalidade["total"] = (
    sazonalidade["internacoes_nao_respiratorias"] + sazonalidade["internacoes_respiratorias"]
)

sazonalidade["pct_respiratorio"] = (
    sazonalidade["internacoes_respiratorias"] / sazonalidade["total"] * 100
)

sazonalidade

RESPIRATORIO,internacoes_nao_respiratorias,internacoes_respiratorias,total,pct_respiratorio
MES_INTER,,,,
1,148414,13865,162279,8.543927
2,142161,13654,155815,8.762956
3,160393,16294,176687,9.221957
4,156737,16337,173074,9.439315
5,159295,18213,177508,10.260383
6,150555,16592,167147,9.926592
7,154213,14679,168892,8.691353
8,158358,14374,172732,8.321562
9,153289,14305,167594,8.535508


A participação das internações respiratórias no total é maior entre abril e junho (outono/início do inverno), com pico em maio, e menor entre outubro e novembro. O padrão é consistente com a sazonalidade esperada de doenças respiratórias no hemisfério sul.

## 6. Municípios com maior demanda respiratória

Assim como no notebook `05_analise_integrada.ipynb`, é utilizado o ano de 2025 como referência, por ser o último ano completo disponível. São consideradas apenas internações de residentes em municípios de Goiás.

In [12]:
df_populacao["codigo_sus_6"] = df_populacao["codigo_ibge_7"].astype("string").str[:6]

populacao_go_2025 = df_populacao[
    (df_populacao["uf"] == "GO") & (df_populacao["ano_referencia"] == 2025)
]

resp_2025 = df_sih_demanda[
    (df_sih_demanda["ANO_INTER"] == 2025) & (df_sih_demanda["RESPIRATORIO"])
]

resp_2025_go = resp_2025[
    resp_2025["MUNIC_RES"].isin(populacao_go_2025["codigo_sus_6"])
]

print("Internações respiratórias em 2025 (residentes em Goiás):", len(resp_2025_go))

Internações respiratórias em 2025 (residentes em Goiás): 44868


### 6.1. Volume absoluto por município

In [13]:
demanda_respiratoria_municipio = (
    resp_2025_go
    .groupby("MUNIC_RES")
    .size()
    .reset_index(name="internacoes_respiratorias")
    .merge(
        populacao_go_2025[["codigo_sus_6", "municipio", "populacao"]],
        left_on="MUNIC_RES",
        right_on="codigo_sus_6",
    )
)

demanda_respiratoria_municipio.sort_values(
    "internacoes_respiratorias", ascending=False
)[["municipio", "internacoes_respiratorias", "populacao"]].head(10)

,municipio,internacoes_respiratorias,populacao
94,Goiânia,5335,1503256
18,Aparecida de Goiânia,2670,556021
197,Rio Verde,2244,241494
15,Anápolis,1382,420300
123,Itumbiara,999,113322
95,Goianira,932,81495
143,Mineiros,869,74999
232,Trindade,776,153560
149,Morrinhos,708,54326
223,Senador Canedo,692,175042


### 6.2. Internações respiratórias por 1.000 habitantes

Para reduzir o ruído de municípios muito pequenos, o ranking por habitante considera apenas municípios com pelo menos 5.000 habitantes — o mesmo tipo de cuidado adotado nas demais análises do projeto ao lidar com bases pequenas.

In [14]:
demanda_respiratoria_municipio["internacoes_por_1000_hab"] = (
    demanda_respiratoria_municipio["internacoes_respiratorias"]
    / demanda_respiratoria_municipio["populacao"]
    * 1000
)

demanda_respiratoria_municipio[
    demanda_respiratoria_municipio["populacao"] >= 5000
].sort_values("internacoes_por_1000_hab", ascending=False)[
    ["municipio", "internacoes_respiratorias", "populacao", "internacoes_por_1000_hab"]
].head(10)

,municipio,internacoes_respiratorias,populacao,internacoes_por_1000_hab
153,Mundo Novo,362,6202,58.368268
177,Paranaiguara,170,7356,23.110386
46,Caiapônia,359,16628,21.590089
142,Minaçu,533,26616,20.025549
170,Ouvidor,142,7665,18.525766
194,Rialma,225,12665,17.765495
62,Cezarina,146,8301,17.588242
178,Paraúna,185,10727,17.246201
225,Silvânia,394,23150,17.019438
33,Bom Jesus de Goiás,417,24925,16.730191


## 7. Mortalidade em internações respiratórias

A mortalidade é comparada entre internações respiratórias e demais internações, utilizando todo o período de análise (2021 a junho de 2026).

In [15]:
df_sih_demanda["MORTE"] = df_sih_demanda["MORTE"].astype("string")

mortalidade = (
    df_sih_demanda
    .groupby("RESPIRATORIO")["MORTE"]
    .apply(lambda serie: (serie == "1").mean() * 100)
    .rename("taxa_mortalidade_pct")
)

mortalidade

RESPIRATORIO
False    3.783036
True     7.981054
Name: taxa_mortalidade_pct, dtype: float64

A taxa de mortalidade das internações respiratórias é mais do que o dobro da taxa das demais internações no período analisado.

## 8. Permanência hospitalar em internações respiratórias

In [16]:
df_sih_demanda["DIAS_PERM_NUM"] = pd.to_numeric(df_sih_demanda["DIAS_PERM"], errors="coerce")

permanencia = (
    df_sih_demanda
    .groupby("RESPIRATORIO")["DIAS_PERM_NUM"]
    .agg(media="mean", mediana="median")
)

permanencia

,media,mediana
RESPIRATORIO,,
False,4.083499,2.0
True,5.247204,3.0


## 9. Uso de UTI em internações respiratórias

In [17]:
df_sih_demanda["UTI_MES_TO_NUM"] = pd.to_numeric(df_sih_demanda["UTI_MES_TO"], errors="coerce")

uso_uti = (
    df_sih_demanda
    .groupby("RESPIRATORIO")
    .apply(lambda grupo: (grupo["UTI_MES_TO_NUM"] > 0).mean() * 100)
    .rename("pct_com_uti")
)

uso_uti

C:\Users\USER\AppData\Local\Temp\ipykernel_8924\1002644016.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grupo: (grupo["UTI_MES_TO_NUM"] > 0).mean() * 100)


RESPIRATORIO
False     8.478851
True     14.081704
Name: pct_com_uti, dtype: float64

## 10. Evasão de município nas internações respiratórias

Uma internação é considerada evasão quando o município de atendimento (`MUNIC_MOV`) é diferente do município de residência do paciente (`MUNIC_RES`), ou seja, o paciente precisou se deslocar para outro município para ser internado.

In [ ]:
df_sih_demanda["MUNIC_MOV"] = df_sih_demanda["MUNIC_MOV"].astype("string").str.zfill(6)

df_sih_demanda["EVASAO"] = (
    df_sih_demanda["MUNIC_RES"] != df_sih_demanda["MUNIC_MOV"]
)

taxa_evasao_geral = (
    df_sih_demanda[df_sih_demanda["RESPIRATORIO"]]["EVASAO"].mean() * 100
)

print(
    "Taxa de evasão das internações respiratórias: {:.2f}%".format(
        taxa_evasao_geral
    )
)

### 10.1. Municípios com maior taxa de evasão

Considera apenas municípios com pelo menos 30 internações respiratórias no período, para reduzir o ruído de municípios com poucos casos.

In [ ]:
evasao_por_municipio = (
    df_sih_demanda[df_sih_demanda["RESPIRATORIO"]]
    .groupby("MUNIC_RES")["EVASAO"]
    .agg(internacoes_respiratorias="size", taxa_evasao="mean")
    .reset_index()
    .merge(
        df_populacao[df_populacao["ano_referencia"] == 2025][
            ["codigo_sus_6", "municipio"]
        ],
        left_on="MUNIC_RES",
        right_on="codigo_sus_6",
        how="left",
    )
)

evasao_por_municipio["taxa_evasao"] = evasao_por_municipio["taxa_evasao"] * 100

evasao_por_municipio[
    evasao_por_municipio["internacoes_respiratorias"] >= 30
].sort_values("taxa_evasao", ascending=False)[
    ["municipio", "internacoes_respiratorias", "taxa_evasao"]
].head(10)

### 10.2. Para onde os pacientes estão sendo levados

Municípios de atendimento (`MUNIC_MOV`) mais frequentes entre os casos de evasão respiratória, no estado inteiro.

In [ ]:
df_sih_demanda[
    df_sih_demanda["RESPIRATORIO"] & df_sih_demanda["EVASAO"]
]["MUNIC_MOV"].value_counts().head(10)

## 11. Caráter da internação e especialidade

`CAR_INT` indica se a internação foi de urgência ou eletiva, e `ESPEC` indica a especialidade em que a internação foi registrada. Os dois campos vêm com o código bruto do SIH/SUS, sem um de-para para texto legível ainda, então os valores aparecem como código, não como descrição.

In [ ]:
df_sih_demanda["CAR_INT"] = df_sih_demanda["CAR_INT"].astype("string")

df_sih_demanda[df_sih_demanda["RESPIRATORIO"]][
    "CAR_INT"
].value_counts(normalize=True).mul(100).round(2)

### 11.1. Especialidade

In [ ]:
df_sih_demanda["ESPEC"] = df_sih_demanda["ESPEC"].astype("string")

df_sih_demanda[df_sih_demanda["RESPIRATORIO"]][
    "ESPEC"
].value_counts().head(10)

## 12. Custo das internações respiratórias

Valor total registrado nas AIHs (`VAL_TOT`), comparando internações respiratórias com as demais.

In [ ]:
df_sih_demanda["VAL_TOT_NUM"] = pd.to_numeric(
    df_sih_demanda["VAL_TOT"], errors="coerce"
)

custo_total_respiratorio = df_sih_demanda[
    df_sih_demanda["RESPIRATORIO"]
]["VAL_TOT_NUM"].sum()

print(
    "Valor total gasto em internações respiratórias no período: "
    "R$ {:,.2f}".format(custo_total_respiratorio)
)

### 12.1. Custo médio por internação

Comparação entre internação respiratória e demais internações.

In [ ]:
custo_medio = (
    df_sih_demanda
    .groupby("RESPIRATORIO")["VAL_TOT_NUM"]
    .mean()
    .rename("valor_medio_aih")
)

custo_medio

## 13. Conclusão

As internações por doenças respiratórias (CID J00-J99) representam cerca de 9% do total de internações regulares em Goiás no período analisado, com participação maior entre abril e junho, coerente com a sazonalidade esperada dessas doenças.

Municípios mais populosos concentram o maior volume absoluto de internações respiratórias, mas municípios menores apresentam taxas por habitante bem mais altas, o que sugere maior pressão relativa sobre a rede local nesses locais.

As internações respiratórias apresentam taxa de mortalidade mais do que o dobro da taxa das demais internações, permanência hospitalar maior (mediana de 3 dias contra 2 dias) e uso de UTI proporcionalmente maior (aproximadamente 14% contra 8%), o que indica que, em média, são internações mais graves que a internação típica da base.

Esta análise não considera idade nem sexo do paciente, pois essas colunas ainda não são extraídas do SIH/SUS no pipeline atual (`src/extract/sih.py`). Para responder perguntas como a concentração de internações respiratórias em crianças ou idosos, é necessário primeiro adicionar essas colunas à extração.